# Designing Layers

EfficientNetB0 is specifically designed for image-based tasks and is highly suitable for facial skin condition analysis. Despite being a deep convolutional neural network, it remains computationally efficient due to its optimized scaling approach. Compared to traditional models, it achieves higher accuracy with fewer parameters, resulting in faster and more efficient training. Therefore, it provides a balance between performance and speed, making it appropriate for this study.

---
## Import Libraries

To Build and connect layers of the CNN model


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
# ── Dataset Parameters ──────────────────────────────────────

IMG_SIZE = 224
# Image size (224x224 pixels)
# Required input size for EfficientNet and other pretrained models

NUM_CLASSES = 5
# Total number of categories (skin diseases) in the dataset
# This must match the number of folders/classes in the data

In [ ]:
# ── Base Model (EfficientNetB0) ─────────────────────────────

base_model = EfficientNetB0(

    weights='imagenet',
    # Load pretrained weights from ImageNet
    # This allows the model to use learned features (edges, textures, patterns)

    include_top=False,
    # Remove the original classification layer of EfficientNet
    # We will add our own custom classifier later

    input_shape=(IMG_SIZE, IMG_SIZE, 3)
    # Input image shape: (height, width, channels)
    # 3 = RGB images

)

In [ ]:
# Initially freeze the base model
base_model.trainable = False

In [ ]:
# ── Custom Classifier Layers ────────────────────────────────

x = base_model.output
# Output from EfficientNet (feature maps)

x = GlobalAveragePooling2D()(x)
# Convert 2D feature maps into a single vector
# Reduces dimensions while keeping important information

x = BatchNormalization()(x)
# Normalizes values to make training more stable and faster

x = Dense(512, activation='relu')(x)
# First dense layer to learn high-level patterns
# ReLU helps avoid vanishing gradient and speeds up learning

x = Dropout(0.4)(x)
# Randomly disables 40% of neurons during training
# Helps prevent overfitting

x = Dense(256, activation='relu')(x)
# Second dense layer to refine learned features

x = Dropout(0.3)(x)
# Additional regularization to improve generalization

output = Dense(NUM_CLASSES, activation='softmax')(x)
# Final output layer
# Softmax converts outputs into probabilities for each class

In [ ]:
model = Model(inputs=base_model.input, outputs=output)

In [ ]:
# ── Fine-tuning Setup ───────────────────────────────────────

# Unfreeze the base model so some layers can be trained
base_model.trainable = True

# Freeze earlier layers, only train the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False
# Earlier layers keep general features (edges, textures)
# Last layers adapt to your specific dataset


# ── Recompile Model ─────────────────────────────────────────
# Required after changing which layers are trainable

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    # Lower learning rate to avoid damaging pretrained weights

    loss='categorical_crossentropy',
    # Suitable for multi-class classification

    metrics=['accuracy']
    # Measure performance
)

In [ ]:
model.summary() # Inspect model